# Tutorial 14A: Building a Simple RNN for Named Entity Recognition (NER) 

### 1. Import libraries

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.preprocessing import LabelEncoder
import numpy as np

# Set random seed for reproducibility
torch.manual_seed(42)
np.random.seed(42)

### 2. Prepare dataset

In [2]:
# 1. Sample data
sentences = [
    "Barack Obama was born in Hawaii",
    "Google is based in Mountain View"
]
labels = [
    ["PERSON", "PERSON", "O", "O", "O", "LOCATION"],
    ["ORGANIZATION", "O", "O", "O", "LOCATION", "O"]
]

# 2. Tokenizing the sentences (converting words into integers)
# Building a simple vocabulary dictionary
vocab = set(word.lower() for sentence in sentences for word in sentence.split())
word2idx = {"<PAD>": 0, "<UNK>": 1} # Adding Padding and Unknown tokens
word2idx.update({word: idx + 2 for idx, word in enumerate(vocab)})

# Convert text to sequences
X = [[word2idx[word.lower()] for word in sentence.split()] for sentence in sentences]

# Padding the sequences to have the same length (post-padding)
max_len = max(len(seq) for seq in X)
X_padded = [seq + [word2idx["<PAD>"]] * (max_len - len(seq)) for seq in X]

# 3. Encode the labels
label_encoder = LabelEncoder()
# We add a <PAD> label to handle the padded positions in the target sequence
label_encoder.fit(["O", "PERSON", "LOCATION", "ORGANIZATION", "<PAD>"])
pad_label_idx = label_encoder.transform(["<PAD>"])[0]

# Convert labels to numerical values
y = [[label_encoder.transform([label])[0] for label in seq] for seq in labels]

# Pad the labels so that they match the shape of the input sequences (X)
y_padded = [seq + [pad_label_idx] * (max_len - len(seq)) for seq in y]

# Convert standard Python lists to PyTorch Tensors
X_tensor = torch.tensor(X_padded, dtype=torch.long)
y_tensor = torch.tensor(y_padded, dtype=torch.long)

print("Input shape:", X_tensor.shape)
print("Target shape:", y_tensor.shape)

Input shape: torch.Size([2, 6])
Target shape: torch.Size([2, 6])


### 3. Build the RNN Model

In [3]:
class SimpleRNN_NER(nn.Module):
    def __init__(self, vocab_size, embedding_dim, rnn_units, num_classes):
        super(SimpleRNN_NER, self).__init__()
        self.embedding = nn.Embedding(num_embeddings=vocab_size, embedding_dim=embedding_dim)
        self.rnn = nn.RNN(input_size=embedding_dim, hidden_size=rnn_units, batch_first=True)
        self.dropout = nn.Dropout(0.1)
        self.fc = nn.Linear(rnn_units, num_classes)

    def forward(self, x):
        x = self.embedding(x)
        rnn_out, _ = self.rnn(x)
        x = self.dropout(rnn_out)
        out = self.fc(x) 
        return out

vocab_size = len(word2idx)
embedding_dim = 50
rnn_units = 50
num_classes = len(label_encoder.classes_)

model = SimpleRNN_NER(vocab_size, embedding_dim, rnn_units, num_classes)

### 4. Compile and Train the Model

In [4]:
# Compile the model equivalents
# ignore_index ensures the loss function doesn't penalize predictions on <PAD> tokens
criterion = nn.CrossEntropyLoss(ignore_index=pad_label_idx)
optimizer = optim.Adam(model.parameters())

# Train the model
epochs = 3

print("Training RNN Model")

for epoch in range(epochs):
    model.train() # Set model to training mode
    
    # 1. Zero the gradients
    optimizer.zero_grad()
    
    # 2. Forward pass
    predictions = model(X_tensor)
    
    # 3. Calculate loss
    # CrossEntropyLoss expects inputs of shape (N, C) and targets of shape (N)
    # We flatten the batch and sequence dimensions using .view(-1)
    loss = criterion(predictions.view(-1, num_classes), y_tensor.view(-1))
    
    # 4. Backward pass (calculate gradients)
    loss.backward()
    
    # 5. Update weights
    optimizer.step()
    
    print(f"Epoch {epoch+1}/{epochs} - Loss: {loss.item():.4f}")
    

Training RNN Model
Epoch 1/3 - Loss: 1.5421
Epoch 2/3 - Loss: 1.5037
Epoch 3/3 - Loss: 1.4291


### 5. Test the Model with a New Sentence

In [5]:
# Test with a new sentence
test_sentence = ["Barack Obama went to Hawaii"]

# Tokenize and pad the test sentence
test_seq = []
for word in test_sentence[0].split():
    # Use <UNK> token (1) if the word was not in the training set
    test_seq.append(word2idx.get(word.lower(), word2idx["<UNK>"]))

# Convert to tensor and add a batch dimension (unsqueeze)
test_tensor = torch.tensor(test_seq, dtype=torch.long).unsqueeze(0) 

# Predicting the NER labels for the test sentence
model.eval() # Set model to evaluation mode
with torch.no_grad():
    predictions = model(test_tensor)
    
    # Get the index of the highest probability class
    predicted_indices = torch.argmax(predictions, dim=-1)[0]

# Decode predictions
decoded_predictions = label_encoder.inverse_transform(predicted_indices.numpy())

# Display results
for word, label in zip(test_sentence[0].split(), decoded_predictions):
    print(f"Word: {word} \t Predicted Label: {label}")

Word: Barack 	 Predicted Label: PERSON
Word: Obama 	 Predicted Label: PERSON
Word: went 	 Predicted Label: PERSON
Word: to 	 Predicted Label: PERSON
Word: Hawaii 	 Predicted Label: <PAD>


### Doesnt predict correct so we will increase the epochs to see if the model predicts correctly or not.


### 6. Trining Improved RNN Model

In [6]:
model_improved = SimpleRNN_NER(vocab_size, embedding_dim, rnn_units, num_classes)

# We use the same loss function, but increase the learning rate slightly
optimizer_improved = optim.Adam(model_improved.parameters(), lr=0.005)

epochs = 100

print(f"Training Improved Model ({epochs} Epochs)...")
for epoch in range(epochs):
    model_improved.train() 
    
    optimizer_improved.zero_grad()
    predictions = model_improved(X_tensor)
    loss = criterion(predictions.view(-1, num_classes), y_tensor.view(-1))
    loss.backward()
    optimizer_improved.step()
    
    if (epoch + 1) % 20 == 0:
        print(f"Epoch {epoch+1}/{epochs} - Loss: {loss.item():.4f}")

Training Improved Model (100 Epochs)...
Epoch 20/100 - Loss: 0.0234
Epoch 40/100 - Loss: 0.0051
Epoch 60/100 - Loss: 0.0017
Epoch 80/100 - Loss: 0.0015
Epoch 100/100 - Loss: 0.0020


### 7. Testing the Imptoved Model

In [7]:
# Test with a new sentence
test_sentence = ["Barack Obama went to Hawaii"]

# Tokenize and pad the test sentence
test_seq = []
for word in test_sentence[0].split():
    # Use <UNK> token (1) if the word was not in the training set
    test_seq.append(word2idx.get(word.lower(), word2idx["<UNK>"]))

# Convert to tensor and add a batch dimension (unsqueeze)
test_tensor = torch.tensor(test_seq, dtype=torch.long).unsqueeze(0) 

# Predicting the NER labels for the test sentence
model_improved.eval() # Set model to evaluation mode
with torch.no_grad():
    predictions_improved = model_improved(test_tensor)
    
    # Get the index of the highest probability class
    predicted_indices_improved = torch.argmax(predictions_improved, dim=-1)[0]

# Decode predictions
decoded_predictions_improved = label_encoder.inverse_transform(predicted_indices_improved.numpy())

# Display results
for word, label in zip(test_sentence[0].split(), decoded_predictions_improved):
    print(f"Word: {word} \t Predicted Label: {label}")

Word: Barack 	 Predicted Label: PERSON
Word: Obama 	 Predicted Label: PERSON
Word: went 	 Predicted Label: O
Word: to 	 Predicted Label: O
Word: Hawaii 	 Predicted Label: LOCATION


### 8. Task 1: Make your own data set and test it

In [8]:
new_rnn_units = 128        # Increased for more complex pattern recognition
new_learning_rate = 0.01   # Increased for faster learning
new_epochs = 50            # Increased to prevent underfitting

print(f"Setting up model with Units: {new_rnn_units}, LR: {new_learning_rate}...")

# Instantiate the new model with updated units
model_combined = SimpleRNN_NER(vocab_size, embedding_dim, new_rnn_units, num_classes)

# Set up the optimizer and loss function
criterion_combined = nn.CrossEntropyLoss(ignore_index=pad_label_idx)
optimizer_combined = optim.Adam(model_combined.parameters(), lr=new_learning_rate)

Setting up model with Units: 128, LR: 0.01...


In [9]:
print(f"Training for {new_epochs} Epochs...\n")

for epoch in range(new_epochs):
    model_combined.train() 
    
    optimizer_combined.zero_grad()
    predictions_combined = model_combined(X_tensor)
    loss_combined = criterion_combined(predictions_combined.view(-1, num_classes), y_tensor.view(-1))
    
    loss_combined.backward()
    optimizer_combined.step()
    
    # Print progress every 10 epochs
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1}/{new_epochs} - Loss: {loss_combined.item():.4f}")

Training for 50 Epochs...

Epoch 10/50 - Loss: 0.0012
Epoch 20/50 - Loss: 0.0001
Epoch 30/50 - Loss: 0.0001
Epoch 40/50 - Loss: 0.0000
Epoch 50/50 - Loss: 0.0000


In [10]:
# Test with Custom Data
custom_sentence = ["I was born in Pakistan"]

# Tokenize the custom sentence
custom_seq = []
for word in custom_sentence[0].split():
    custom_seq.append(word2idx.get(word.lower(), word2idx["<UNK>"]))

# Convert to tensor and add a batch dimension
custom_tensor = torch.tensor(custom_seq, dtype=torch.long).unsqueeze(0) 
print("Custom data prepared and tokenized!")

Custom data prepared and tokenized!


In [11]:
# Predict using the evaluation mode
model_combined.eval()
with torch.no_grad():
    custom_predictions = model_combined(custom_tensor)
    
    # Get the highest probability class indices
    custom_predicted_indices = torch.argmax(custom_predictions, dim=-1)[0]

# Decode the numerical predictions back into text labels (e.g., PERSON, LOCATION)
custom_decoded = label_encoder.inverse_transform(custom_predicted_indices.numpy())

# Display Results
print("Custom Sentence Predictions:")
for word, label in zip(custom_sentence[0].split(), custom_decoded):
    print(f"Word: {word} \t Predicted Label: {label}")

Custom Sentence Predictions:
Word: I 	 Predicted Label: O
Word: was 	 Predicted Label: O
Word: born 	 Predicted Label: O
Word: in 	 Predicted Label: O
Word: Pakistan 	 Predicted Label: LOCATION
